In [1]:
from google.colab import files
uploaded = files.upload()

Saving archive (1).zip to archive (1).zip


In [2]:
import zipfile
import os

zip_path = "archive (1).zip"  # Replace with actual filename
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [3]:
os.listdir("/content/dataset")

['fertilizer_recommendation.csv']

In [6]:
import pandas as pd

df = pd.read_csv("/content/dataset/fertilizer_recommendation.csv")
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Temperature,Humidity,Rainfall,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Previous_Crop,Region,Fertilizer_Used_Last_Season,Yield_Last_Season,Recommended_Fertilizer
0,Clay,6.07,34.98,0.32,1.87,61,44,84,19.84,83.31,1693.22,Cotton,Harvest,Kharif,Canal,Wheat,South,297.15,1.19,MOP
1,Silt,6.39,47.34,0.28,0.21,59,56,18,24.40,46.27,1030.21,Maize,Vegetative,Kharif,Sprinkler,Potato,Central,77.17,4.40,Urea
2,Sandy,7.92,38.13,0.99,1.88,43,21,119,24.82,71.86,1166.16,Cotton,Flowering,Kharif,Rainfed,Tomato,South,128.93,7.21,Urea
3,Clay,5.86,14.17,1.46,0.36,88,46,34,27.87,53.23,2881.83,Wheat,Flowering,Zaid,Sprinkler,Potato,West,233.96,1.85,MOP
4,Clay,7.98,19.28,0.85,2.16,104,53,98,24.17,51.87,714.84,Potato,Sowing,Kharif,Rainfed,Maize,East,214.39,7.36,Zinc Sulphate


In [7]:
!pip install lightgbm optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 28.1 MB/s eta 0:00:00


In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

import lightgbm as lgb
import optuna

In [9]:
df = pd.read_csv("/content/dataset/fertilizer_recommendation.csv")
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Temperature,Humidity,Rainfall,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Previous_Crop,Region,Fertilizer_Used_Last_Season,Yield_Last_Season,Recommended_Fertilizer
0,Clay,6.07,34.98,0.32,1.87,61,44,84,19.84,83.31,1693.22,Cotton,Harvest,Kharif,Canal,Wheat,South,297.15,1.19,MOP
1,Silt,6.39,47.34,0.28,0.21,59,56,18,24.40,46.27,1030.21,Maize,Vegetative,Kharif,Sprinkler,Potato,Central,77.17,4.40,Urea
2,Sandy,7.92,38.13,0.99,1.88,43,21,119,24.82,71.86,1166.16,Cotton,Flowering,Kharif,Rainfed,Tomato,South,128.93,7.21,Urea
3,Clay,5.86,14.17,1.46,0.36,88,46,34,27.87,53.23,2881.83,Wheat,Flowering,Zaid,Sprinkler,Potato,West,233.96,1.85,MOP
4,Clay,7.98,19.28,0.85,2.16,104,53,98,24.17,51.87,714.84,Potato,Sowing,Kharif,Rainfed,Maize,East,214.39,7.36,Zinc Sulphate


In [10]:
TARGET = "Recommended_Fertilizer"

In [11]:
DROP_COLS = [
    "Yield_Last_Season",
    "Fertilizer_Used_Last_Season"
]

df = df.drop(columns=DROP_COLS, errors="ignore")

In [12]:
categorical_features = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Previous_Crop",
    "Region"
]

numerical_features = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Nitrogen_Level",
    "Phosphorus_Level",
    "Potassium_Level",
    "Temperature",
    "Humidity",
    "Rainfall"
]

In [13]:
# Numerical → median
for col in numerical_features:
    df[col] = df[col].fillna(df[col].median())

# Categorical → mode
for col in categorical_features:
    df[col] = df[col].fillna(df[col].mode()[0])

In [14]:
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [15]:
df["NPK_Ratio"] = (
    df["Nitrogen_Level"] +
    df["Phosphorus_Level"] +
    df["Potassium_Level"]
) / 3

In [16]:
df["Moisture_Rain_Interaction"] = df["Soil_Moisture"] * df["Rainfall"]

In [17]:
df["Climate_Stress"] = df["Temperature"] / (df["Humidity"] + 1)

In [22]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
df[TARGET] = target_encoder.fit_transform(df[TARGET])

df[TARGET].head()

,Recommended_Fertilizer
0,2
1,5
2,5
3,2
4,6


In [23]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [30]:
def objective(trial):

    params = {
        "objective": "multiclass",
        "num_class": y.nunique(),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "verbosity": -1,

        # 🔥 REQUIRED FOR OPTUNA + min_data_in_leaf
        "feature_pre_filter": False,

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0)
    }

    # 🔥 CREATE DATASET INSIDE OBJECTIVE
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_test, label=y_test)

    model = lgb.train(
        params,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=300,
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )

    preds = model.predict(X_test)
    preds = np.argmax(preds, axis=1)

    return accuracy_score(y_test, preds)

In [31]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

study.best_value, study.best_params

[I 2026-02-16 17:14:34,236] A new study created in memory with name: no-name-6ba244c7-14ff-4740-b3cb-ea3aee89e1b3
[I 2026-02-16 17:14:35,388] Trial 0 finished with value: 0.875 and parameters: {'learning_rate': 0.14643270644639444, 'num_leaves': 197, 'max_depth': 9, 'min_data_in_leaf': 11, 'feature_fraction': 0.9311312451023719, 'bagging_fraction': 0.6932082601891683, 'bagging_freq': 6, 'lambda_l1': 2.9789049452766267, 'lambda_l2': 2.996319993016871}. Best is trial 0 with value: 0.875.
[I 2026-02-16 17:14:37,467] Trial 1 finished with value: 0.873 and parameters: {'learning_rate': 0.08303389566167103, 'num_leaves': 225, 'max_depth': 10, 'min_data_in_leaf': 11, 'feature_fraction': 0.6663171895525756, 'bagging_fraction': 0.9856274068295166, 'bagging_freq': 2, 'lambda_l1': 0.3024909985106683, 'lambda_l2': 0.6155396724779555}. Best is trial 0 with value: 0.875.
[I 2026-02-16 17:14:38,142] Trial 2 finished with value: 0.8735 and parameters: {'learning_rate': 0.1464807607755581, 'num_leaves'

(0.878,
 {'learning_rate': 0.12604441436118582,
  'num_leaves': 271,
  'max_depth': 6,
  'min_data_in_leaf': 23,
  'feature_fraction': 0.7224739583229978,
  'bagging_fraction': 0.84022207207405,
  'bagging_freq': 6,
  'lambda_l1': 3.8848271764275366,
  'lambda_l2': 4.859940132586908})

In [32]:
study.best_value
study.best_params

{'learning_rate': 0.12604441436118582,
 'num_leaves': 271,
 'max_depth': 6,
 'min_data_in_leaf': 23,
 'feature_fraction': 0.7224739583229978,
 'bagging_fraction': 0.84022207207405,
 'bagging_freq': 6,
 'lambda_l1': 3.8848271764275366,
 'lambda_l2': 4.859940132586908}

In [33]:
best_params = study.best_params
best_params.update({
    "objective": "multiclass",
    "num_class": y.nunique(),
    "metric": "multi_logloss",
    "verbosity": -1
})

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=500
)

In [34]:
study.best_value

0.878

In [35]:
from sklearn.metrics import accuracy_score

y_pred = final_model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy Score:", accuracy)

Accuracy Score: 0.8745


In [36]:
import joblib

joblib.dump(final_model, "fertilizer_model.pkl")
joblib.dump(target_encoder, "target_encoder.pkl")
joblib.dump(label_encoders, "feature_encoders.pkl")

['feature_encoders.pkl']

In [49]:
import numpy as np
import pandas as pd

def predict_fertilizer(new_data_dict):

    # Convert input dictionary to DataFrame
    new_df = pd.DataFrame([new_data_dict])

    # Apply same feature engineering
    new_df["NPK_Ratio"] = (
        new_df["Nitrogen_Level"] +
        new_df["Phosphorus_Level"] +
        new_df["Potassium_Level"]
    ) / 3

    new_df["Moisture_Rain_Interaction"] = (
        new_df["Soil_Moisture"] * new_df["Rainfall"]
    )

    new_df["Climate_Stress"] = (
        new_df["Temperature"] / (new_df["Humidity"] + 1)
    )

    for col in categorical_features:
       if new_df[col][0] in label_encoders[col].classes_:
          new_df[col] = label_encoders[col].transform(new_df[col])
       else:
          new_df[col] = 0  # fallback for unseen category

    #  Predict
    prediction = final_model.predict(new_df)
    prediction = np.argmax(prediction, axis=1)

    #  Convert back to fertilizer name
    fertilizer_name = target_encoder.inverse_transform(prediction)

    return fertilizer_name[0]

In [53]:
new_farmer_data = {
    "Soil_Type": "hehe",
    "Soil_pH": 6.5,
    "Soil_Moisture": 35,
    "Organic_Carbon": 0.8,
    "Electrical_Conductivity": 0.3,
    "Nitrogen_Level": 50,
    "Phosphorus_Level": 40,
    "Potassium_Level": 45,
    "Temperature": 28,
    "Humidity": 65,
    "Rainfall": 120,
    "Crop_Type": "Wheat",
    "Crop_Growth_Stage": "Vegetative",
    "Season": "Rabi",
    "Irrigation_Type": "Drip",
    "Previous_Crop": "Rice",
    "Region": "North"
}

In [54]:
predict_fertilizer(new_farmer_data)

'Urea'